# Datagen

Generate realistic Delta tables and a Direct Lake semantic model from a Power BI `.vpax` file.

## Setup

1. **Attach a Lakehouse** — click the Lakehouse icon in the left sidebar and select (or create) a Lakehouse
2. **Upload your `.vpax` file** — place it in `Files/datagen/` in the attached Lakehouse
   - Export a `.vpax` from [DAX Studio](https://daxstudio.org/) → Advanced → Export Metrics
3. **Run Cell 1** below to install datagen (auto-downloads from GitHub if needed)
4. **Edit Cell 2** — change the `.vpax` filename to match yours
5. **Run Cell 2** — generates Delta tables, deploys a semantic model, and prints a comparison report


In [ ]:
# Install datagen — downloads the latest release from GitHub if not cached locally
import glob, subprocess, sys, os, urllib.request, json

DATAGEN_DIR = "/lakehouse/default/Files/datagen"
os.makedirs(DATAGEN_DIR, exist_ok=True)

whls = sorted(glob.glob(f"{DATAGEN_DIR}/datagen_fabric-*.whl"))
if whls:
    whl = whls[-1]
else:
    api_url = "https://api.github.com/repos/dbrownems/datagen/releases/latest"
    with urllib.request.urlopen(api_url) as resp:
        release = json.loads(resp.read())
    asset = next(a for a in release["assets"] if a["name"].endswith(".whl"))
    whl = f"{DATAGEN_DIR}/{asset['name']}"
    print(f"Downloading {asset['name']} ...")
    urllib.request.urlretrieve(asset["browser_download_url"], whl)

print(f"Installing {os.path.basename(whl)}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", whl, "semantic-link-labs",
    "-q", "--no-warn-conflicts", "--disable-pip-version-check",
])
print("Ready.")

## Generate

Edit the `.vpax` filename below, then run the cell. This will:

1. **Parse** the `.vpax` and infer data distributions from column statistics
2. **Generate** Delta tables matching the original row counts and cardinality
3. **Deploy** a Direct Lake semantic model with all measures, relationships, and column metadata
4. **Compare** the generated tables against the expected statistics and print a report

### Options

| Parameter | Default | Description |
|---|---|---|
| `mode` | `"direct_lake"` | `"import"` for Power Query import mode |
| `deploy_model` | `True` | `False` to skip semantic model deployment |
| `compare` | `True` | `False` to skip comparison report |
| `overwrite` | `False` | `True` to overwrite an existing semantic model |
| `seed` | `42` | Random seed for reproducible generation |


In [ ]:
from datagen import generate

VPAX_FILE = "AdventureWorks.vpax"  # ← change this to your .vpax filename

report = generate(
    spark,
    f"{DATAGEN_DIR}/{VPAX_FILE}",
    overwrite=True,
)